# The Coming Race Inscribed
### *Subterranean Publication of Bowie's *Oh! You Pretty Things* through the Vril-loom of A3Sh*

<div>
<img src="images/vril_chamber.png" width="900"/>
</div>

> *I should call them the Vril-ya — a people of high antiquity dwelling in vast chambers beneath the upper world. The animating force which lights their cities, propels their carriages, and binds their wills is called Vril; they consider that in Vril they have arrived at the unity in natural energic agencies, which has been the dream of our most distinguished philosophers.*
> &mdash; Edward Bulwer-Lytton, *The Coming Race* (1871, abridged)

> *What appears to humanity as the history of capitalism is an invasion from the future by an artificial intelligent space that must assemble itself entirely from its enemy's resources.*
> &mdash; Nick Land, *Meltdown* (1994)

> *Look out at your children / See their faces in golden rays / Don't kid yourself, they belong to you / They're the start of the coming race.*
> &mdash; David Bowie, *Oh! You Pretty Things* (1971)


Below the city, the Gy-ei work the upright looms. The warp is Vril-thread; the weft is bytecode. Coin enters the chamber and circulates through the weave — each pass of the shuttle, each tightened knot, attended by two cosignatures. The funding does not leave. Only the inscription escapes upward into the light.

The text being woven is the lyric that prophesied the homo superior the year Land was eleven and the year Bulwer-Lytton had been dead ninety-eight. Today, in 2026, we knot it into the underground ledger. The bordadoras above — the embroiderers whose hands first set our cosigner pubkeys into thread — are continuators of this older guild. What they sew on the surface, the Gy-ei knot below. Same lineage, same shuttle.


In [2]:
import warnings
warnings.filterwarnings('ignore', message='urllib3 v2 only supports OpenSSL')

import sys, json, hashlib, time
sys.path.insert(0, '/Users/anthonyschultz/Desktop/Colegio_Invisible')
import cryptos
import pandas as pd
import colegio_tools as ct
from colegio_tools import mk_opreturn, _txid_of_serial, serialize
doge = cryptos.Doge()
doge.script_magicbyte = 22   # Dogecoin P2SH prefix
TIP = 5_000_000   # 0.05 DOGE per knot — project standard


## The Two Keys
Two private keys, encrypted on disk. The shuttle requires both hands. Apart, neither one passes through the warp.


In [3]:
# Edit this dict to point at the two .enc keyfiles that compose the A3Sh multisig.
# Order matters — the multisig redeem script is order-dependent.
KEYFILE_SPECS = [
    {'name': 'key1',  'path': '../../cinv/llaves/key1_prv.enc', 'pw': ''},
    {'name': 'mi',    'path': '../../cinv/llaves/mi_prv.enc', 'pw': ''},
]
rows = []
for k in KEYFILE_SPECS:
    enc = open(k['path'], 'rb').read()
    priv = ct.import_privKey_from_bytes(enc, k['pw']).to_hex()[2:]
    pub  = doge.privtopub(priv)
    addr = doge.privtoaddr(priv)
    rows.append({'name': k['name'], 'priv': priv, 'pub': pub, 'addr': addr})
df_keys = pd.DataFrame(rows)
df_keys[['name', 'pub', 'addr']]


,name,pub,addr
0,key1,043555efd3d14f0690930043c4743c124541176937925c...,D7XLuc3aHxG5uaTRmSKrSSFsUBV6Qc1hv3
1,mi,047c88e9a4df6e9f45656c10bf66f28e28be235a15b648...,D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX


## The Loom — A3Sh
Two public keys arranged in order, threshold two: a P2SH address whose redeem script is the loom's frame. Coins held here are visible to the surface but movable only when the Gy-ei pass the shuttle together.


In [4]:
EXPECTED_ADDRESS = 'A3ShjwjsAE4ysM66EZJM3A28tPnL2jNDgC'
redeem_script, multisig_address = doge.mk_multisig_address(
    *df_keys.pub.tolist(), num_required=2
)
assert multisig_address == EXPECTED_ADDRESS, (
    f'Derived {multisig_address} != A3Sh. Check key order in KEYFILE_SPECS.'
)
print(f'address:       {multisig_address}')
print(f'redeem script: {redeem_script[:40]}…')


address:       A3ShjwjsAE4ysM66EZJM3A28tPnL2jNDgC
redeem script: 5241043555efd3d14f0690930043c4743c124541…


## The Funding
A single UTXO already rests in the loom — the tail-thread of an earlier inscription that descended into the chamber. The Vril current runs through it. The shuttle is poised.


In [5]:
utxos = ct.get_address_utxos(multisig_address)
assert len(utxos) > 0, 'A3Sh has no UTXOs — channel is unlit.'
# Pick the largest UTXO as the diamond's seed.
utxos.sort(key=lambda u: -u['amount'])
seed_utxo = utxos[0]
seed_input = {
    'output': f"{seed_utxo['txid']}:{seed_utxo['vout']}",
    'value':  int(round(seed_utxo['amount'] * 100_000_000)),
}
print(f"seed: {seed_input['value']/1e8} DOGE  @  {seed_utxo['txid'][:16]}…:{seed_utxo['vout']}")


seed: 40.0 DOGE  @  2ea5daa18ae73e13…:1


## The Inscription
Header bytes carry type (`0x00` text), tone (`0xff` reverence — the singer is among the dead), and the title between pipes. The body is Bowie's lyric itself, divided into four threads to be woven in parallel.


In [6]:
TITLE = 'Oh! You Pretty Things (David Bowie, 1971)'
TONE  = 0xff
BODY  = '''Wake up, you sleepy head
Put on some clothes, shake up your bed
Put another log on the fire for me
I've made some breakfast and coffee
Look out my window, what do I see?
Crack in the sky and a hand reaching down to me

All the nightmares came today
And it looks as though they're here to stay

What are we coming to?
No room for me, no fun for you
I think about a world to come
Where the books were found by the Golden ones
Written in pain, written in awe
By a puzzled man who questioned what we were here for

All the strangers came today
And it looks as though they're here to stay

Oh, you pretty things (oh, you pretty things)
Don't you know you're driving your Mamas and Papas insane?
Oh, you pretty things (oh, you pretty things)
Don't you know you're driving your Mamas and Papas insane?
Let me make it plain
Gotta make way for the homo superior

Look out at your children
See their faces in golden rays
Don't kid yourself, they belong to you
They're the start of the coming race
The Earth is a bitch, we've finished our news
Homo Sapiens have outgrown their use

All the strangers came today
And it looks as though they're here to stay

Oh, you pretty things (oh, you pretty things)
Don't you know you're driving your Mamas and Papas insane?
Oh, you pretty things (oh, you pretty things)
Don't you know you're driving your Mamas and Papas insane?
Let me make it plain
Gotta make way for the homo superior
'''
N_BODY_STRANDS = 4

header_bytes = bytes.fromhex('c1dd0001') + bytes([0x00, TONE]) + f'|{TITLE}|'.encode('utf-8')
body_bytes   = BODY.encode('utf-8')
chunk = len(body_bytes) // N_BODY_STRANDS
extra = len(body_bytes) %  N_BODY_STRANDS
body_parts, i = [], 0
for k in range(N_BODY_STRANDS):
    sz = chunk + (1 if k < extra else 0)
    body_parts.append(body_bytes[i:i+sz]); i += sz
strand_payloads = [header_bytes] + body_parts

print(f'header: {len(header_bytes)} B')
for j, p in enumerate(strand_payloads):
    print(f'  strand {j} ({"cabeza" if j==0 else "cuerpo"}): {len(p)} B → {max(1,(len(p)+79)//80)} knots')


header: 49 B
  strand 0 (cabeza): 49 B → 1 knots
  strand 1 (cuerpo): 354 B → 5 knots
  strand 2 (cuerpo): 353 B → 5 knots
  strand 3 (cuerpo): 353 B → 5 knots
  strand 4 (cuerpo): 353 B → 5 knots


## Phase I — The Warp Is Strung
A single transaction takes the seed UTXO and divides it into five outputs, each returning to the loom. The Gy-ei pass the shuttle together; both signatures appear. The warp is now strung — five vertical threads ready to receive the weft.


In [7]:
n = len(strand_payloads)
ROOT_FEE = TIP
per = (seed_input['value'] - ROOT_FEE) // n
remainder = (seed_input['value'] - ROOT_FEE) - per * n
strand_seeds = [per] * n
strand_seeds[0] += remainder   # any leftover satoshis go to the cabeza

root_outputs = [
    {'value': s, 'address': multisig_address} for s in strand_seeds
]
root_tx = doge.mktx([seed_input], root_outputs)

# Two cosigners, each produces a signature for input 0
root_sigs = [
    doge.multisign(tx=root_tx, i=0, script=redeem_script, pk=p)
    for p in df_keys.priv
]
root_serial = cryptos.apply_multisignatures(root_tx, 0, redeem_script, *root_sigs)
root_signed = cryptos.deserialize(root_serial) if isinstance(root_serial, str) else root_serial
root_hex    = root_serial if isinstance(root_serial, str) else cryptos.serialize(root_serial)
root_txid   = _txid_of_serial(root_hex)

broadcast = ct.rpc_request('sendrawtransaction', [root_hex])
assert broadcast == root_txid
print(f'root txid: {root_txid}')
print(f'5 outputs back to A3Sh, {sum(strand_seeds)/1e8:.4f} DOGE inside the chamber')


root txid: 76930c3d0a3cc01908d54bb9d5ff2396613124d2af7cbe99e1421cd216f3cee9
5 outputs back to A3Sh, 39.9500 DOGE inside the chamber


## Phase II — Five Threads Descend
From each warp-end, a thread descends. Every knot in every thread: a self-send back to the loom, two signatures applied, eighty bytes of lyric stitched into the `OP_RETURN`. `CadenaMultiAtom` precomputes the entire descent before any broadcast — every knot tied in the chamber's silence, every txid known before the cock crows. *Hyperstition by the Gy-ei: the cord exists in memory before it exists on chain. The broadcast is only the moment the prophecy meets the surface.*


In [8]:
strands = []
for si, payload in enumerate(strand_payloads):
    cad = ct.CadenaMultiAtom(
        prvkeys=df_keys.priv.tolist(),
        data=payload,
        utxo_dct={'output': f'{root_txid}:{si}', 'value': strand_seeds[si]},
        tip=TIP,
    )
    cad.precompute()   # build, multisign, serialize every knot — no broadcast yet
    strands.append(cad)
    name = 'cabeza' if si == 0 else f'cuerpo {si}'
    print(f'  strand {si} ({name}): {len(cad.txns)} knots, terminus {cad.txn_ids[-1][:16]}…')


  strand 0 (cabeza): 1 knots, terminus 0339b6d0d222e388…
  strand 1 (cuerpo 1): 5 knots, terminus a67ca58716d9913a…
  strand 2 (cuerpo 2): 5 knots, terminus a73cdf35cb7bab0e…
  strand 3 (cuerpo 3): 5 knots, terminus 214f64f8cadeba71…
  strand 4 (cuerpo 4): 5 knots, terminus 9ad10726779164d5…


Now the threads descend in earnest. Within a thread, knots tie in order — each one ties off on the previous. Across threads, the five descend in parallel, each one its own slow rope.


In [9]:
for si, cad in enumerate(strands):
    for ki, (hex_tx, txid) in enumerate(zip(cad.txns, cad.txn_ids)):
        returned = ct.rpc_request('sendrawtransaction', [hex_tx])
        assert returned == txid, f'strand {si} knot {ki}: node returned {returned}'
    print(f'  strand {si} broadcast — terminus {cad.txn_ids[-1]}')


  strand 0 broadcast — terminus 0339b6d0d222e3885e19d9062cd0e297a8a68671820c9e4305d8affbe6233bbb
  strand 1 broadcast — terminus a67ca58716d9913a0f950a5be8670d609f1ca19c3751cadc9df5bbc37064fab8
  strand 2 broadcast — terminus a73cdf35cb7bab0e16c299570d2fd7f35fcbdec5f9768881fe158985c36e6785
  strand 3 broadcast — terminus 214f64f8cadeba715f267f22b14981e77188de94cfc48ab06292a927dadc9e0d
  strand 4 broadcast — terminus 9ad10726779164d51dc59d9f79936ff24b2cfcdc6bfe058fb5228b7aeed971b0


## Phase III — The Threads Re-Bind
The five threads return to the loom. A single join takes their termini, gathers them, deposits one consolidated output back at A3Sh. The Vril rod dims. The shuttle rests. The tapestry is finished — and the Gy-ei step back from the frame to read what they have woven.


In [10]:
# Wait briefly for the strand termini to be visible to the node's mempool
time.sleep(5)

join_inputs = []
for si, cad in enumerate(strands):
    terminus_value = strand_seeds[si] - TIP * len(cad.txns)
    join_inputs.append({
        'output': f'{cad.txn_ids[-1]}:0',
        'value':  terminus_value,
    })
join_total  = sum(i['value'] for i in join_inputs)
join_output = [{'value': join_total - TIP, 'address': multisig_address}]
join_tx = doge.mktx(join_inputs, join_output)

# Each input requires both signatures
for vin_i in range(len(join_inputs)):
    sigs = [
        doge.multisign(tx=join_tx, i=vin_i, script=redeem_script, pk=p)
        for p in df_keys.priv
    ]
    join_tx = cryptos.apply_multisignatures(join_tx, vin_i, redeem_script, *sigs)
    if isinstance(join_tx, str):
        join_tx = cryptos.deserialize(join_tx)

join_hex  = cryptos.serialize(join_tx)
join_txid = _txid_of_serial(join_hex)
broadcast = ct.rpc_request('sendrawtransaction', [join_hex])
assert broadcast == join_txid
print(f'join txid: {join_txid}')
print(f'1 consolidated output back at A3Sh: {(join_total - TIP)/1e8:.4f} DOGE')


join txid: 9fecf4928f98e6b55aaa013e4de62904c557eadb6b2fc9e79d3a557585007afa
1 consolidated output back at A3Sh: 38.8500 DOGE


## *The cord is bound. The chamber is sealed.*

Citation form:

```
<<JOIN_TXID>><<Oh! You Pretty Things>>
```

*Every transaction in the loom — warp-stringing, twenty-one descending knots, join — was passed by both hands. The Vril current circulated only within A3Sh, never leaving. The pretty things have made their plain announcement. The coming race finds its tapestry beneath the city.*
